## Bronze Table: Routes

Ingests data from the `routes` volume and appends it to the `bronze\routes` Delta table with the date and timestamp of ingestion.

Columns containing only null values are dropped to prevent schema conflicts.

In [0]:
import pandas as pd
from datetime import datetime

In [0]:

file_path = "/Volumes/trips_carris_metropolitana/bronze/routes/routes.json"
df = pd.read_json(file_path)

# Drop entirely null columns (programmatically)
df = df.dropna(axis=1, how='all')

# Convert empty arrays to None to avoid NullType inference
for col in df.columns:
    if df[col].apply(lambda x: isinstance(x, list) and len(x) == 0).all():
        df[col] = ""

# Add ingestion metadata
df["ingestion_date"] = datetime.now().strftime("%Y-%m-%d")
df["ingestion_timestamp"] = datetime.now()

# Create Spark DataFrame (now all columns are simple types)
spark_df = spark.createDataFrame(df)
spark_df.write.mode("append").saveAsTable("trips_carris_metropolitana.bronze.routes")

In [0]:
%sql
SELECT COUNT(*) FROM trips_carris_metropolitana.bronze.routes;

In [0]:
%sql
SELECT * FROM trips_carris_metropolitana.bronze.routes LIMIT 5;